# Decision Tree — Scheme 2 (Static Test) — GAMEEMO

> Run on **Google Colab**. Mount your Google Drive and adjust `folder_path` before executing.


In [ ]:
import os, numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ── Define training size (change per experiment step) ──
train_size = 0.95  # fraction of training pool used at this step

folder_path   = '/content/drive/My Drive/EEG Datasets/GAMEEMO/Denoised EEG Data/'
test_file_path = folder_path + 'gameemo_test.csv'

subject_files = [[f"S{i:02}_G{j}_Denoised.csv" for j in range(1, 5)] for i in range(1, 29)]
data_list = []
for subject_file_list in subject_files:
    for file in subject_file_list:
        temp_data = pd.read_csv(os.path.join(folder_path, file))
        data_list.append(temp_data)

train_pool = pd.concat(data_list, ignore_index=True)
train_pool = train_pool.sample(frac=0.10, random_state=42).reset_index(drop=True)

# ── Load the FIXED static test set (never subsampled) ──
test_data = pd.read_csv(test_file_path)

# ── Subsample training pool ──
data_train = train_pool.sample(frac=train_size, random_state=42).reset_index(drop=True)

X_train_raw = data_train.drop(columns=['Valence', 'Arousal']).values
y_train_raw = data_train['Valence'].values
X_test_raw  = test_data.drop(columns=['Valence', 'Arousal']).values
y_test_raw  = test_data['Valence'].values

# ── Fit scaler and encoder on TRAINING data ONLY ──
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

valence_enc = LabelEncoder()
y_train = valence_enc.fit_transform(y_train_raw)
y_test  = valence_enc.transform(y_test_raw)

# ────────────────────────────────────────────────────────────
# Decision Tree — Scheme 2 (Static Test) — GAMEEMO
# ────────────────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(criterion='gini', splitter='best', max_depth=10, random_state=42)

# ── 5-Fold Cross-Validation on training data ──
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(f"5-Fold CV Accuracy with {int(train_size * 100)}% training data: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} (SD)")

model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)
print(f"Held-out Test Accuracy with {int(train_size * 100)}% training data: {accuracy:.4f}")

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred, normalize='true')
plt.figure(figsize=(5, 4))
sns.heatmap(cm * 100, annot=True, fmt=".2f", cmap="Blues")
plt.title(f"Confusion Matrix ({int(train_size * 100)}% Training Data)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()